In [1]:
from dotenv import load_dotenv
load_dotenv()
import os 

from typing import List
from pydantic import BaseModel, Field
import uuid

from langgraph.graph import START, END, StateGraph, MessagesState
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage

from langgraph.store.postgres import PostgresStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig


In [2]:
# ----------------------------
# 2) System prompt
# ----------------------------
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user's preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible.

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""

In [3]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
memory_llm = ChatGroq(model="openai/gpt-oss-20b")

In [4]:
class MemoryItem(BaseModel):
    text: str = Field(description="Atomic user memory")
    is_new: bool = Field(description="True if new, false if duplicate")

In [5]:
class MemoryDecision(BaseModel):
    should_write: bool 
    memories: List[MemoryItem] = Field(default_factory=list)

In [6]:
memory_extractor = memory_llm.with_structured_output(MemoryDecision)

In [7]:
MEMORY_PROMPT = """You are responsible for updating and maintaining accurate user memory.

CURRENT USER DETAILS (existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specific info worth storing long-term (identity, stable preferences, ongoing projects/goals).
- For each extracted item, set is_new=true ONLY if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false.
- Keep each memory as a short atomic sentence.
- No speculation; only facts stated by the user.
- If there is nothing memory-worthy, return should_write=false and an empty list.
"""

In [8]:
def remember_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config['configurable']['user_id']
    ns = ('user', user_id, 'details')

    items = store.search(ns)
    existing = '\n'.join(it.value.get('data', '') for it in items) if items else '(empty)'

    last_text = state['messages'][-1].content 

    decision: MemoryDecision = memory_extractor.invoke([
        SystemMessage(content=MEMORY_PROMPT.format(user_details_content=existing)),
        {"role": "user", "content": last_text}
    ])

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new and mem.text.strip():
                store.put(ns, str(uuid.uuid4()), {'data': mem.text.strip()})

    return {}


In [9]:
chat_llm = ChatGroq(model="openai/gpt-oss-20b")

In [10]:
def chat_node(state: MessagesState, config: RunnableConfig, * , store: BaseStore):
    user_id = config['configurable']['user_id']
    ns = ('user', user_id, 'details')

    items = store.search(ns)
    user_details = ''.join(itm.value.get('data', '') for itm in items) if items else ''

    system_message = SystemMessage(
        content=SYSTEM_PROMPT_TEMPLATE.format(user_details_content=user_details or '(empty)')
    )
    response = chat_llm.invoke([system_message] + state['messages'])

    return {'messages': [response]}

In [11]:
# ----------------------------
# 4) Build graph
# ---------------------------
builder = StateGraph(MessagesState)
builder.add_node("remember", remember_node)
builder.add_node("chat", chat_node)
builder.add_edge(START, "remember")
builder.add_edge("remember", "chat")
builder.add_edge("chat", END)

In [12]:
with PostgresStore.from_conn_string(os.getenv('DATABASE_URL')) as store:
    store.setup() 
    graph = builder.compile(store=store) 
    config = {'configurable': {'user_id': 'user1'}}

    graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Indrasis"}]}, config)
    graph.invoke({"messages": [{"role": "user", "content": "I am a Frontend heavy AI-powered MERN stack developer"}]}, config)

    output = graph.invoke({
        'messages': [
            {'role': 'user', 'content': 'Explain Gen AI simply'}
        ],        
    }, config)
    print(output['messages'][-1].content)


    print("\n--- Stored Memories (from Postgres) ---")

    for it in store.search(("user", "user1", "details")):
        print(it.value["data"])

Hey Indrasis! 👋

### What is Generative AI (Gen AI) in a nutshell?

Think of Gen AI as a super‑creative version of a code‑autocompletion tool—except it can generate *content* instead of just snippets.  
- **Input → Model → Output**  
  1. **You give it a prompt** (e.g., “Write me a short product description for a smartwatch”).  
  2. **The model processes the prompt** using patterns it learned from a huge dataset of text, images, or code.  
  3. **It spits out a new piece of content** that matches the style and intent you asked for.

**Key points:**
- **Generative** – it creates, not just predicts the next word.  
- **AI‑powered** – uses deep learning (usually transformer architectures like GPT, DALL‑E, or Stable Diffusion).  
- **Versatile** – can write text, code, design mockups, generate audio, synthesize images, etc.

#### Why it matters for a MERN developer

| Task | Gen AI can help |
|------|-----------------|
| **Front‑end UI** | Auto‑generate CSS/React snippets, suggest compone

In [13]:
items = store.search(('user', 'user1', 'details'))

print(f"Total memories: {len(items)}")

for item in items:
    print(item.value['data'])

OperationalError: the connection is closed